In [ ]:
# =====================================================================
# Algorithm Comparison: NSGA-II vs AGE-MOEA vs MOEA/D
# Same physical engine (KDE+multi-height+self-blockage).
# Only the algorithm changes — Problem class stays identical.
# =====================================================================

import torch, numpy as np, math, time
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.age import AGEMOEA
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.termination import get_termination
from pymoo.optimize import minimize

device = torch.device('cuda'); print(f'Device: {device}')

# ============================================================
# 1. Physical Engine (same as pareto_front.ipynb)
# ============================================================
rx,ry,rz=10.0,10.0,3.0; xBs,yBs,zBs=10.0,-100.0,-10.0
E=8; dB=0.075; lam=0.075; L1=2; dref=abs(yBs)*1.5
Pd=40.0; Rth=0.2; Nd=-174.0; Bw=20e6; pm_val=0.2; nu=5
PB=10**(Pd/10)*1e-3; N0=10**(Nd/10)*1e-3*Bw

# RWP generator (for KDE)
def gen_rwp_traj(sim_time,dt=10,nu=5,rx=10.0,ry=10.0,rng=None):
    if rng is None: rng=np.random
    rs=[0.0,rx,0.0,ry]; hc=np.array([rx/2,ry/2]); hr=rx/3.0
    p_t=0.6;p_s=0.3;tau_h=1.5;tau_n=0.1;v_h=0.2;v_n=1.0;g_h=0.6;g_n=0.2
    ts=int(sim_time/dt)
    def gt(p):
        t=hc+(rng.rand(2)-0.5)*2*hr if rng.rand()<g_h else np.array([rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])])
        t[0]=np.clip(t[0],rs[0],rs[1]);t[1]=np.clip(t[1],rs[2],rs[3]);return t
    def ta(r):return tau_h if r=='hot'else tau_n
    def sp(r):return v_h if r=='hot'else v_n
    uh=1.5+0.5*rng.rand(nu);pos=np.zeros((nu,ts,2))
    up=np.zeros((nu,2));ur=[None]*nu;us=[None]*nu;ut_=np.zeros((nu,2));ud=np.zeros((nu,2));usp=np.zeros(nu);uprem=np.zeros(nu)
    for i in range(nu):
        up[i]=[rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])];d2h=np.linalg.norm(up[i]-hc);ur[i]='hot'if d2h<=hr else'normal'
        if rng.rand()<p_t:us[i]='transfer';ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
        else:us[i]='pause';uprem[i]=rng.exponential(ta(ur[i]))
        pos[i,0,:]=up[i]
    for step in range(1,ts):
        for i in range(nu):
            if us[i]=='pause':
                uprem[i]-=dt;pos[i,step,:]=up[i]
                if uprem[i]<=0:us[i]='transfer';ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
            else:
                md=usp[i]*dt;pd=ut_[i]-up[i]
                if np.linalg.norm(pd)<=md:
                    up[i]=ut_[i].copy();d2h=np.linalg.norm(up[i]-hc);ur[i]='hot'if d2h<=hr else'normal'
                    if rng.rand()<p_s:us[i]='pause';uprem[i]=rng.exponential(ta(ur[i]))
                    else:ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
                else:up[i]=up[i]+ud[i]*md
                pos[i,step,:]=up[i]
    pts=np.zeros((nu*ts,3));idx=0
    for u in range(nu):
        for s in range(ts):pts[idx]=[pos[u,s,0],pos[u,s,1],uh[u]];idx+=1
    return pts

# Multi-height grid + KDE weights
GR=200; Z_HS=[1.5,1.625,1.75,1.875,2.0]; N_Z=len(Z_HS)
xv=torch.linspace(0,rx,GR,device=device); yv=torch.linspace(0,ry,GR,device=device)
Xg,Yg=torch.meshgrid(xv,yv,indexing='ij'); xyf=torch.stack([Xg.flatten(),Yg.flatten()],dim=1)
gl=[]; [gl.append(torch.stack([Xg.flatten(),Yg.flatten(),torch.full_like(Xg.flatten(),zh)],dim=1)) for zh in Z_HS]
gl=torch.cat(gl,dim=0); Ngrid=gl.shape[0]
np.random.seed(777); et=gen_rwp_traj(sim_time=100000,dt=10)
kde=gaussian_kde(et[:,:2].T,bw_method='scott')
wxy=kde(xyf.cpu().numpy().T).flatten(); wxy=wxy/wxy.sum()
gw=torch.tensor(np.tile(wxy,N_Z),dtype=torch.float32,device=device); gw=gw/gw.sum()
np.random.seed(42)
_ps=torch.tensor(2*np.pi*np.random.rand(Ngrid),dtype=torch.float32,device=device)
_tt=torch.tensor(-np.pi+2*np.pi*np.random.rand(Ngrid),dtype=torch.float32,device=device)
_eta=torch.tensor(10**((-15+5*np.random.rand(Ngrid))/10),dtype=torch.float32,device=device)
print(f'Grid: {N_Z}x{GR}x{GR}={Ngrid} pts, hotspot/edge={gw.max().item()/gw.min().item():.1f}x')

# Static tensors
_n_ant=torch.arange(E,dtype=torch.float32,device=device)
_dy_BS=torch.tensor(0.0-yBs,device=device); _v1c=lam/(4*math.pi); _v1pc=-(2*math.pi/lam); _oE=1/math.sqrt(E)

# Channel model
def phys_chan(wp,lc):
    if not isinstance(wp,torch.Tensor): wp=torch.tensor(wp,dtype=torch.float32,device=device)
    if wp.dim()==1: wp=wp.unsqueeze(0)
    Bn=wp.shape[0]; xc,zc,Lx,Lz=wp[:,0],wp[:,1],wp[:,2],wp[:,3]
    xu,yu,zu=lc[:,0],lc[:,1],lc[:,2]
    dx=xc-xBs; dy=_dy_BS; dz=zc-zBs
    Rw=torch.sqrt(dx**2+dy**2+dz**2); th=torch.atan2(dy,dx); ph=torch.acos(dz/Rw)
    kx=torch.sin(ph)*torch.cos(th); kz=torch.cos(ph)
    x1=xc-Lx/2;z1=zc-Lz/2;x2=xc-Lx/2;z2=zc+Lz/2;x3=xc+Lx/2;z3=zc-Lz/2;x4=xc+Lx/2;z4=zc+Lz/2
    def rd(xs,zs):
        ddx=xs.unsqueeze(1)-xBs;ddy=_dy_BS;ddz=zs.unsqueeze(1)-zBs
        L=torch.sqrt(ddx**2+ddy**2+ddz**2);return ddx/L,ddy/L,ddz/L
    ux1,_,uz1=rd(x1,z1);ux2,_,uz2=rd(x2,z2);ux3,_,uz3=rd(x3,z3);ux4,_,uz4=rd(x4,z4)
    dux=xu-xBs;duy=yu-yBs;duz=zu-zBs;Lu=torch.sqrt(dux**2+duy**2+duz**2)
    uux=dux/Lu;uuz=duz/Lu
    ua=torch.stack([ux1,ux2,ux3,ux4],0);uz=torch.stack([uz1,uz2,uz3,uz4],0)
    umin=ua.min(0).values;umax=ua.max(0).values;zmin=uz.min(0).values;zmax=uz.max(0).values
    b=1000.0
    ix=torch.sigmoid(b*(uux-umin))*torch.sigmoid(b*(umax-uux))
    iz=torch.sigmoid(b*(uuz-zmin))*torch.sigmoid(b*(zmax-uuz))
    il=ix*iz
    d2x=xu-xc.unsqueeze(1);d2y=yu;d2z=zu-zc.unsqueeze(1)
    Ru=torch.sqrt(d2x**2+d2y**2+d2z**2);t1=d2x/Ru;t2=d2z/Ru
    ax=(1-il)*(kx.unsqueeze(1)-t1);az=(1-il)*(kz.unsqueeze(1)-t2)
    sx=torch.sinc((math.pi/lam)*Lx.unsqueeze(1)*ax);sz=torch.sinc((math.pi/lam)*Lz.unsqueeze(1)*az)
    p1=(2*math.pi/lam)*dB*_n_ant*torch.sin(th).unsqueeze(1)
    a1=_oE*torch.complex(torch.cos(p1),torch.sin(p1))
    v1m=_v1c/Rw;v1p=_v1pc*Rw
    v1=torch.complex(v1m*torch.cos(v1p),v1m*torch.sin(v1p));H1=v1.unsqueeze(1)*a1.conj()
    p2=(2*math.pi/lam)*dB*_n_ant.unsqueeze(0)*torch.sin(_tt).unsqueeze(1)
    a2=_oE*torch.complex(torch.cos(p2),torch.sin(p2))
    v2m=_eta*(lam/(4*math.pi*dref));v2=torch.complex(v2m*torch.cos(_ps),v2m*torch.sin(_ps))
    H2=v2.unsqueeze(1)*a2.conj().unsqueeze(0)
    Ht=math.sqrt(E/L1)*(H1.unsqueeze(1)+H2)
    fm=(Lx.unsqueeze(1)*Lz.unsqueeze(1)*sx*sz)/(lam*Ru)
    fp=(-(2*math.pi/lam)*(kx*xc+kz*zc)-(math.pi/2))
    fc=torch.complex(fm*torch.cos(fp.unsqueeze(1)),fm*torch.sin(fp.unsqueeze(1)))
    return Ht*fc.unsqueeze(2)

# Grid outage evaluator
def compute_grid_outage(xc,zc,Lx,Lz):
    theta=torch.tensor([xc,zc,Lx,Lz],dtype=torch.float32,device=device)
    He=phys_chan(theta,gl); Hw=torch.sum(He,dim=2)/math.sqrt(E)
    dp=(torch.abs(Hw)**2)*pm_val*PB; it=(nu-1)*dp; sr=dp/(it+N0)
    # Self-blockage
    ab=math.pi/3.0; Ses=0
    rn=torch.sqrt((gl[:,0]-xc)**2+(gl[:,2]-zc)**2+gl[:,1]**2+1e-12)
    for a in [0.0,math.pi/2,math.pi,3*math.pi/2]:
        dot=torch.abs((gl[:,0]-xc)*math.cos(a)+gl[:,1]*math.sin(a))
        cp=dot/rn; ph=torch.acos(torch.clamp(cp,0,1))
        Se=(math.pi-torch.clamp(2*ab-ph,min=0))/math.pi
        Ses=Ses+torch.clamp(Se,1/3,1)
    sr=((Ses/4)*dp)/((Ses/4)*it+N0)
    bits=(torch.log2(1+sr)<Rth).float()
    return float(torch.sum(bits*gw).item())

print(f'Warm-up: [5,1.5,0.3,0.3] outage={compute_grid_outage(5.0,1.5,0.3,0.3)*100:.2f}%\n')

# ============================================================
# 2. Shared Problem Class
# ============================================================
lb=np.array([0.2,0.2,0.1,0.1]); ub=np.array([9.8,2.8,9.8,2.8])

class EMWindowProblem(ElementwiseProblem):
    def __init__(self):
        super().__init__(n_var=4,n_obj=2,n_ieq_constr=4,xl=lb,xu=ub)
    def _evaluate(self,x,out,*a,**k):
        xc,zc,Lx,Lz=x
        out["G"]=[Lx/2-xc,xc+Lx/2-rx,Lz/2-zc,zc+Lz/2-rz]
        out["F"]=[Lx*Lz,compute_grid_outage(xc,zc,Lx,Lz)]

class EMWindowProblemNoConstr(ElementwiseProblem):
    """For MOEA/D which does not support constraints — use penalty"""
    def __init__(self):
        super().__init__(n_var=4,n_obj=2,xl=lb,xu=ub)
    def _evaluate(self,x,out,*a,**k):
        xc,zc,Lx,Lz=x
        viol = max(0,Lx/2-xc)+max(0,xc+Lx/2-rx)+max(0,Lz/2-zc)+max(0,zc+Lz/2-rz)
        out["F"]=[Lx*Lz,compute_grid_outage(xc,zc,Lx,Lz)+viol*0.5]

# ============================================================
# 3. Run Three Algorithms
# ============================================================
POP,GEN=300,200
results={}

for name,algo in [
    ('NSGA-II', NSGA2(pop_size=POP,n_offsprings=150,sampling=FloatRandomSampling(),
        crossover=SBX(prob=0.9,eta=15),mutation=PM(prob=0.25,eta=20),eliminate_duplicates=True)),
    ('AGE-MOEA', AGEMOEA(pop_size=POP,n_offsprings=150,sampling=FloatRandomSampling(),
        crossover=SBX(prob=0.9,eta=15),mutation=PM(prob=0.25,eta=20))),
    ('MOEA/D', MOEAD(ref_dirs=get_reference_directions('das-dennis',2,n_partitions=20),
        n_neighbors=20,prob_neighbor_mating=0.9,
        crossover=SBX(prob=0.9,eta=15),mutation=PM(prob=0.25,eta=20))),  # no constraints supported, penalty in objective
]:
    print(f'\n{"="*50}\n {name} (pop={POP}, gen={GEN})\n{"="*50}')
    t0=time.time()
    prob=EMWindowProblemNoConstr() if 'MOEA/D' in name else EMWindowProblem()
    res=minimize(prob,algo,get_termination('n_gen',GEN),seed=42,verbose=True)
    elapsed=time.time()-t0
    results[name]={'F':res.F,'X':res.X,'time':elapsed}
    feasible=res.F[:,1]<=0.10
    if feasible.any():
        bi=np.where(feasible)[0][np.argmin(res.F[feasible,0])]
        print(f'Knee: {res.F[bi,0]:.4f} m² @ {res.F[bi,1]*100:.2f}% | time={elapsed:.0f}s')
    else:
        print(f'No feasible solution | time={elapsed:.0f}s')

# ============================================================
# 4. Comparison Plot
# ============================================================
fig,ax=plt.subplots(figsize=(10,6))
colors={'NSGA-II':'darkgreen','AGE-MOEA':'darkorange','MOEA/D':'steelblue'}

for name,res in results.items():
    F=res['F']; idx=np.argsort(F[:,1]); F=F[idx]
    ax.plot(F[:,1]*100,F[:,0],'o-',color=colors[name],markersize=4,linewidth=1.5,label=f'{name} ({len(F)} pts)')
    # Knee
    feas=F[:,1]<=0.10
    if feas.any():
        bi=np.argmin(F[feas,0]); ba,bo=F[feas,0][bi],F[feas,1][bi]
        ax.plot(bo*100,ba,'*',color=colors[name],markersize=16,markeredgewidth=1.5)
        ax.annotate(f'{ba:.3f}m²',(bo*100,ba),fontsize=8,color=colors[name],fontweight='bold')

ax.axvline(x=10,color='red',ls='--',alpha=0.5,label='10% outage'); ax.legend(fontsize=10)
ax.set_xlabel('Grid Outage [%]',fontsize=12); ax.set_ylabel('Window Area [m²]',fontsize=12)
ax.set_title(f'Algorithm Comparison: NSGA-II vs AGE-MOEA vs MOEA/D',fontsize=14); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('algo_comparison.png',dpi=150); plt.show()

# ============================================================
# 5. Summary Table
# ============================================================
print('\n'+'='*72)
print(f'{"Algorithm":<12} {"Knee Area":<12} {"Outage":<10} {"Pareto pts":<12} {"Time":<10}')
print('-'*56)
for name,res in results.items():
    F=res['F']; feas=F[:,1]<=0.10
    if feas.any():
        bi=np.argmin(F[feas,0]); ka,ko=F[feas,0][bi],F[feas,1][bi]
        print(f'{name:<12} {ka:<12.4f} {ko*100:<10.2f}% {len(F):<12} {res["time"]:<10.0f}s')
    else:
        print(f'{name:<12} {"N/A":<12} {"N/A":<10} {len(F):<12} {res["time"]:<10.0f}s')
print('='*72)

from IPython.display import FileLink,display
display(FileLink('algo_comparison.png'))